<a href="https://colab.research.google.com/github/teoteoh/sctec_aulas/blob/main/Miniprojeto_TeodoraCosta_Analise_de_Dados_TI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importando as bibliotecas relevantes:

In [ ]:
import csv
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

Carregando CSV e fazendo as primeiras
inspeções:

In [ ]:
df = pd.read_csv('Base Varejo.csv', sep=';')

display(df.head())


In [ ]:
print(df.info())

Há colunas sem rótulo e com valores vazios. Fazemos a remoção das colunas vazias para manter a base de dados mais limpa e concisa:

In [ ]:
df = df.dropna(axis=1, how='all')

Checamos para ver se há linhas duplicadas:

In [ ]:
if df.duplicated().sum() > 0:
    print(f"Linhas duplicadas: {df.duplicated().sum()}")
else:
    print("\nNão há duplicatas.")

Agora removemos duplicatas e checamos o resultado:

In [ ]:
df = df.drop_duplicates()

print(df.info())

Transformando a coluna DATA em DataTime:

In [ ]:
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y')

display(df.head())
print(df['DATA'].dtype)

Agora verificamos se há categorias vazias:

In [ ]:
print("NaN por coluna:")
print(df.isna().sum())

print("Strings por coluna:")
for col in df.select_dtypes(include=['object']).columns:
    vazios = (df[col].astype(str).str.strip() == '').sum()
    if vazios > 0:
        print(f"{col}: {vazios} registros vazios.")
    else:
        print(f"{col}: Nenhum registro vazio.")

Substituimos os NaN por "Sem Categoria" e checamos o resultado:

In [ ]:
#df = df.fillna('Sem Categoria')

#print(f"Valores vazios: {df.isna().sum().sum()}")

Estatísticas básicas relacionadas ao número de filhos dos clientes:

In [ ]:
stats = df['CL_FHL'].describe()
mode_val = df['CL_FHL'].mode()[0]
median_val = df['CL_FHL'].median()

print("Estatísticas Descritivas Referentes ao Número de Filhos:")
print(f"Contagem:        {stats['count']}")
print(f"Média:           {stats['mean']:.2f}")
print(f"Mediana:         {median_val}")
print(f"Moda:            {mode_val}")
print(f"Desvio Padrão:   {stats['std']:.2f}")
print(f"Mínimo:          {stats['min']}")
print(f"Máximo:          {stats['max']}")
print(f"Quartil 25%:     {stats['25%']}")
print(f"Quartil 50%:     {stats['50%']}")
print(f"Quartil 75%:     {stats['75%']}")

Agora agrupamos os principais produtos por gênero:

In [ ]:
produtos_por_genero = df.groupby(['CL_GENERO', 'PR_NOME']).size().reset_index(name='QUANTIDADE')
grupos = produtos_por_genero.groupby('CL_GENERO')
top_10_por_genero = pd.concat([grupo.nlargest(10, 'QUANTIDADE') for _, grupo in grupos]).reset_index(drop=True)

plt.figure(figsize=(12, 8))
sns.barplot(data=top_10_por_genero, x='QUANTIDADE', y='PR_NOME', hue='CL_GENERO')

plt.title('Top 10 Produtos Mais Comprados por Gênero')
plt.xlabel('Quantidade Comprada')
plt.ylabel('Produto')
plt.legend(title='Gênero')
plt.tight_layout()
plt.show()

Agrupamos os principais produtos por segmento social:

In [ ]:
produtos_por_segmento = df.groupby(['CL_SEG', 'PR_NOME']).size().reset_index(name='QUANTIDADE')

grupos_seg = produtos_por_segmento.groupby('CL_SEG')
top_10_por_segmento = pd.concat([grupo.nlargest(10, 'QUANTIDADE') for _, grupo in grupos_seg]).reset_index(drop=True)

plt.figure(figsize=(14, 10))
sns.barplot(data=top_10_por_segmento, x='QUANTIDADE', y='PR_NOME', hue='CL_SEG')

plt.title('Top 10 Produtos Mais Comprados por Segmento Social')
plt.xlabel('Quantidade Comprada')
plt.ylabel('Produto')
plt.legend(title='Segmento Social')
plt.tight_layout()
plt.show()

Agora agrupamos por gênero baseado no segmento social:

In [ ]:
prod_seg_gen = df.groupby(['CL_SEG', 'CL_GENERO', 'PR_NOME']).size().reset_index(name='QUANTIDADE')

top_prod_comparativo = prod_seg_gen.groupby(['CL_SEG', 'CL_GENERO']).apply(lambda x: x.nlargest(5, 'QUANTIDADE')).reset_index(drop=True)

g = sns.catplot(
    data=top_prod_comparativo,
    kind="bar",
    x="QUANTIDADE", y="PR_NOME",
    hue="CL_GENERO", col="CL_SEG",
    palette="muted", height=6, aspect=0.8
)

g.set_titles("Segmento Social: {col_name}")
g.set_axis_labels("Quantidade", "Produto")
plt.subplots_adjust(top=0.9)
g.fig.suptitle('Comparativo de Top Produtos: Homens vs Mulheres por Segmento Social')
plt.show()

Insights com base nos gráficos gerados:

1.   Preseunto cozido e sardinha são dois produtos que tanto homens quanto mulheres compram em quase a mesma quantidade, independente do segmento social. Aumentar a oferta e a variedade desses produtos pode gerar um lucro maior para o negócio.
2.   Os principais produtos comprados por mulheres tendem a incluir mais produtos de higiene e limpeza do que produtos comprados por homens. Isso indica que os papéis de gênero influenciam significativamente as compras, e que mulheres ainda são as principais responsáveis pela limpeza do ambiente.
3.  Percebe-se que mulheres da classe social C compram mais produtos de limpeza do que mulheres da classe social A e B. Isso indica que mulheres de classes sociais mais altas não são responsáveis por limpeza. Oferecer promoções e ofertas de produtos de limpeza pode deixar os protudos mais acessíveis ao principal segmento que os consome.
4.  Homens do segmento social C são responsáveis por adquirir produtos de higiene para bebês. Isso indica que o gráfico se passa em uma realidade alternativa em que homens são os principais cuidadores dos filhos durante os primeiros anos de suas vidas. Ou que, no caso de casais, são os homens que arcam com as despesas relacionadas aos bebês.

